In [3]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import pyarrow.parquet as pq

In [5]:
pwd

'/content'

In [6]:
train = pd.read_csv('shift_ml_2026_train.csv')

/tmp/ipython-input-4264814569.py:1: DtypeWarning: Columns (49) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv('shift_ml_2026_train.csv')


In [ ]:
train.head()

,id,сумма_займа,срок_займа,процентная_ставка,аннуитет,рейтинг,допрейтинг,профессия_заемщика,стаж,владение_жильем,...,процент_счетов_прев_75_лимита,кол-во_публ_банкротств,кол-во_залогов,кредитный_лимит,кредитный_баланс_без_ипотеки,лимит_по_картам,лимит_по_аннуитетным_счетам,кредитный_баланс_по_возоб_счетам,особая_ситуация,тип_предоставления_кредита
0,68355089,1235000.00,3 года,11.99,41014.00,В,В1,инженер,10+ лет,ИПОТЕКА,...,7.70,0.00,0.00,15700850.00,1973750.00,3965000.00,1233350.00,NaN,Нет,Наличные
1,68341763,1000000.00,5 лет,10.78,21633.00,Б,Б4,водитель грузовика,10+ лет,ИПОТЕКА,...,50.00,0.00,0.00,10920900.00,934800.00,310000.00,743850.00,NaN,Нет,Наличные
2,68426831,597500.00,3 года,13.44,20259.00,В,В3,ветеринарный техник,4 года,АРЕНДА,...,100.00,0.00,0.00,845000.00,639900.00,470000.00,200000.00,NaN,Нет,Наличные
3,68476668,1000000.00,3 года,9.17,31879.00,Б,Б2,вице-президент операций по набору персонала,10+ лет,ИПОТЕКА,...,100.00,0.00,0.00,19442600.00,5838100.00,1575000.00,2322600.00,NaN,Нет,Наличные
4,67275481,1000000.00,3 года,8.49,31563.00,Б,Б1,дорожному водителю,10+ лет,ИПОТЕКА,...,0.00,0.00,0.00,9669500.00,1396850.00,725000.00,1807200.00,NaN,Нет,Наличные


In [ ]:
train.shape

(1210779, 109)

In [ ]:
train.duplicated().sum()

np.int64(0)

In [7]:
limit = len(train) * 0.7   # удалить колонки, где более 30% пропусков - оставила те, где хотя бы 70% данных (0.7) заполнены
train = train.dropna(thresh=limit, axis=1)

In [8]:
train.shape

(1210779, 84)

In [ ]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1210779 entries, 0 to 1210778
Data columns (total 84 columns):
 #   Column                                             Non-Null Count    Dtype  
---  ------                                             --------------    -----  
 0   id                                                 1210779 non-null  int64  
 1   сумма_займа                                        1210779 non-null  float64
 2   срок_займа                                         1210779 non-null  object 
 3   процентная_ставка                                  1210779 non-null  float64
 4   аннуитет                                           1210779 non-null  float64
 5   рейтинг                                            1210779 non-null  object 
 6   допрейтинг                                         1210779 non-null  object 
 7   профессия_заемщика                                 1133730 non-null  object 
 8   стаж                                               1140271 non

In [9]:
train = train.fillna(train.median(numeric_only=True))   # пропуски в числовых признаках заполняем медианой

In [ ]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1210779 entries, 0 to 1210778
Data columns (total 84 columns):
 #   Column                                             Non-Null Count    Dtype  
---  ------                                             --------------    -----  
 0   id                                                 1210779 non-null  int64  
 1   сумма_займа                                        1210779 non-null  float64
 2   срок_займа                                         1210779 non-null  object 
 3   процентная_ставка                                  1210779 non-null  float64
 4   аннуитет                                           1210779 non-null  float64
 5   рейтинг                                            1210779 non-null  object 
 6   допрейтинг                                         1210779 non-null  object 
 7   профессия_заемщика                                 1133730 non-null  object 
 8   стаж                                               1140271 non

In [10]:
train['профессия_заемщика'] = train['профессия_заемщика'].fillna('unknown')
train['стаж'] = train['стаж'].fillna('unknown')        # остались пропуски в двух кат признаках, заполняем 'unknown'

In [ ]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1210779 entries, 0 to 1210778
Data columns (total 84 columns):
 #   Column                                             Non-Null Count    Dtype  
---  ------                                             --------------    -----  
 0   id                                                 1210779 non-null  int64  
 1   сумма_займа                                        1210779 non-null  float64
 2   срок_займа                                         1210779 non-null  object 
 3   процентная_ставка                                  1210779 non-null  float64
 4   аннуитет                                           1210779 non-null  float64
 5   рейтинг                                            1210779 non-null  object 
 6   допрейтинг                                         1210779 non-null  object 
 7   профессия_заемщика                                 1210779 non-null  object 
 8   стаж                                               1210779 non

удалила колонки, где более 30% пропусков - остались те, где хотя бы 70% данных. В них уже числовые пропуски заменила медианой, а в кат. на 'unknown'

CatBoostClassifier - библиотека машинного обучения, работает намного лучше и быстрее логистической регрессии, особенно на «грязных» данных с текстом

In [18]:
from catboost import CatBoostClassifier

In [13]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 9.5 MB/s eta 0:00:00


In [11]:
object_cols = train.select_dtypes(include=['object']).columns.tolist()
print(object_cols)

['срок_займа', 'рейтинг', 'допрейтинг', 'профессия_заемщика', 'стаж', 'владение_жильем', 'подтвержден_ли_доход', 'платежный_график', 'цель_займа', 'регион', 'пос_стоп_фактор', 'юридический_статус', 'дата_первого_займа', 'первоначальный_статус_займа', 'тип_займа', 'особая_ситуация', 'тип_предоставления_кредита']


In [12]:
x_train = train.drop('итоговый_статус_займа', axis=1) # на основе чего предсказываем
y_train = train['итоговый_статус_займа']              # то, что предсказываем

In [ ]:
# Указываем для модели, какие колонки категориальные
cat_features = object_cols

model = CatBoostClassifier(
    iterations=1000,            # количество деревьев. Большее значение может повысить точность, но увеличивает время обучения.
    learning_rate=0.1,          # темп обучения (0.01–0.1) Меньшее значение позволяет модели учиться более плавно и уменьшает риск переобучения
    depth=6,                    # глубина деревьев регулирует сложность модели. Глубокие деревья могут лучше выявлять сложные зависимости, но также рискуют переобучиться.
    cat_features=cat_features,  # набор данных
    verbose=100                 # показывать прогресс каждые 100 шагов
)

model.fit(x_train, y_train)

0:	learn: 0.6385291	total: 3.96s	remaining: 1h 5m 58s
100:	learn: 0.4256864	total: 6m 38s	remaining: 59m 7s
200:	learn: 0.4219462	total: 13m 1s	remaining: 51m 46s
300:	learn: 0.4195624	total: 19m 30s	remaining: 45m 18s
400:	learn: 0.4179284	total: 25m 59s	remaining: 38m 49s
500:	learn: 0.4166474	total: 32m 36s	remaining: 32m 29s
600:	learn: 0.4154633	total: 39m 6s	remaining: 25m 57s
700:	learn: 0.4144463	total: 45m 22s	remaining: 19m 21s
800:	learn: 0.4135944	total: 51m 36s	remaining: 12m 49s
900:	learn: 0.4127532	total: 58m 1s	remaining: 6m 22s
999:	learn: 0.4119718	total: 1h 4m 19s	remaining: 0us


In [ ]:
model.classes_

array([0, 1])

In [14]:
from sklearn.model_selection import cross_val_score

In [ ]:
# Проводим кросс-валидацию (например, на 5 фолдах)

scores = cross_val_score(estimator=model,          # Ваша модель
                         X=x_train,               # Признаки (отмасштабированные)
                         y=y_train,                      # Целевая переменная
                         cv=5,                     # Количество фолдов (чаще всего 3, 5 или 10)
                         scoring='roc_auc'        # Используемая метрика
                         )

0:	learn: 0.6398909	total: 3.45s	remaining: 57m 22s
100:	learn: 0.4279935	total: 5m 12s	remaining: 46m 21s
200:	learn: 0.4238820	total: 10m 17s	remaining: 40m 55s
300:	learn: 0.4213354	total: 15m 22s	remaining: 35m 42s
400:	learn: 0.4196004	total: 20m 25s	remaining: 30m 30s
500:	learn: 0.4181065	total: 25m 35s	remaining: 25m 29s
600:	learn: 0.4169489	total: 30m 38s	remaining: 20m 20s
700:	learn: 0.4158457	total: 35m 40s	remaining: 15m 12s
800:	learn: 0.4147376	total: 40m 46s	remaining: 10m 7s
900:	learn: 0.4137394	total: 45m 59s	remaining: 5m 3s
999:	learn: 0.4128211	total: 50m 56s	remaining: 0us
0:	learn: 0.6392846	total: 2.65s	remaining: 44m 8s
100:	learn: 0.4202641	total: 5m 22s	remaining: 47m 54s
200:	learn: 0.4163116	total: 10m 23s	remaining: 41m 18s
300:	learn: 0.4136775	total: 15m 30s	remaining: 36m
400:	learn: 0.4119406	total: 20m 45s	remaining: 31m
500:	learn: 0.4105998	total: 25m 50s	remaining: 25m 44s
600:	learn: 0.4093635	total: 30m 56s	remaining: 20m 32s
700:	learn: 0.4081

In [ ]:
# 1. результаты всех 5 тестов
print("Результаты на каждом фолде:", scores)

# 2. среднее значение
print(f"Средний ROC-AUC: {scores.mean():.4f}")

# 3. разброс (стабильность модели)
print(f"Отклонение: +/- {scores.std():.4f}")

Результаты на каждом фолде: [0.78370105 0.69779473 0.60676137 0.64455091 0.69860338]
Средний ROC-AUC: 0.6863
Отклонение: +/- 0.0597


In [ ]:
model.save_model('loan_model_v1.cbm') # сохранить модель
# 1. Создаем "пустой" объект класса
# new_model = CatBoostClassifier()

# 2. Загружаем в него сохраненные данные. Не нужно заново указывать cat_features, модель их уже «запомнила» внутри файла
# new_model.load_model('loan_model_v1.cbm')

**зачем снова обучать модель fit для поиска важности, если она обучена еще до cross_val_score?**
Когда вы передаете модель в cross_val_score(model, X, y), происходит следующее:
 - Система берет ваш «шаблон» модели (объект model).
 - Копирует его 5 раз.
 - Обучает каждую копию на своем кусочке данных и удаляет их после вычисления метрики.
 - Ваш исходный объект model остается пустым (необученным).
Если вы прямо сейчас попробуете вызвать model.get_feature_importance(), вы получите ошибку: "NotFittedError: Call fit before querying feature importances".

In [ ]:
# Обучаем быстро, просто чтобы заглянуть "в голову" модели
# 1. Создаем новый объект с нужными параметрами
model = CatBoostClassifier(
    iterations=200,
    verbose=50,
    cat_features=cat_features,
    auto_class_weights='Balanced' # чтобы подтянуть ROC-AUC с 0.68
)

# 2. Теперь fit сработает без ошибок
model.fit(x_train, y_train)

Learning rate set to 0.5
0:	learn: 0.6286761	total: 6.71s	remaining: 22m 14s
50:	learn: 0.5752528	total: 3m 43s	remaining: 10m 52s
100:	learn: 0.5690827	total: 7m 5s	remaining: 6m 56s
150:	learn: 0.5649243	total: 10m 27s	remaining: 3m 23s
199:	learn: 0.5616694	total: 13m 53s	remaining: 0us


In [ ]:
# 3. Достаем важность
importances = model.get_feature_importance(type='FeatureImportance')
feature_importance_df = pd.DataFrame({'Feature': x_train.columns, 'Importance': importances}).sort_values(by='Importance', ascending=False)
print(feature_importance_df)

                                           Feature  Importance
5                                          рейтинг       19.14
36                      сумма_выплат_по_просрочкам       11.38
6                                       допрейтинг        8.86
0                                               id        7.66
7                               профессия_заемщика        4.89
..                                             ...         ...
41              кол-во_текущих_просроченных_счетов        0.00
23               коэфф_невыплаченного_сумм_остатка        0.00
69  кол-во_счетов_с_акт_просрочкой_4_мес_обн_2_мес        0.00
81                                 особая_ситуация        0.00
82                      тип_предоставления_кредита        0.00

[83 rows x 2 columns]


In [ ]:
feature_importance_df[feature_importance_df.Importance==0].Feature.tolist()

['рейтинг_OCR',
 'платежный_график',
 'первоначальный_статус_займа',
 'юридический_статус',
 'кол-во_списаний_за_год',
 'непогашенная_сумма_из_тела_займов',
 'код_политики',
 'кол-во_текущих_просроченных_счетов',
 'коэфф_невыплаченного_сумм_остатка',
 'кол-во_счетов_с_акт_просрочкой_4_мес_обн_2_мес',
 'особая_ситуация',
 'тип_предоставления_кредита']

In [15]:
# Удаляем "мусорные" признаки из данных
x_train_clean = x_train.drop(columns=['id',
 'рейтинг_OCR',
 'платежный_график',
 'первоначальный_статус_займа',
 'юридический_статус',
 'кол-во_списаний_за_год',
 'непогашенная_сумма_из_тела_займов',
 'код_политики',
 'кол-во_текущих_просроченных_счетов',
 'коэфф_невыплаченного_сумм_остатка',
 'кол-во_счетов_с_акт_просрочкой_4_мес_обн_2_мес',
 'особая_ситуация',
 'тип_предоставления_кредита'])

In [16]:
# обновляю список кат. признаков
cat_features_clean  = x_train_clean.select_dtypes(include=['object']).columns.tolist()
print(cat_features_clean )

['срок_займа', 'рейтинг', 'допрейтинг', 'профессия_заемщика', 'стаж', 'владение_жильем', 'подтвержден_ли_доход', 'цель_займа', 'регион', 'пос_стоп_фактор', 'дата_первого_займа', 'тип_займа']


In [19]:
# Создаем модель с балансировкой весов и очищенными данными
model_check = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05, # Чуть снижаем, чтобы лучше обучилась
    cat_features=cat_features_clean,
    auto_class_weights='Balanced', # учитываем дисбаланс классов
    verbose=100
)

Почему мы не делаем fit прямо сейчас?
Чтобы не тратить ресурсы. Сначала мы должны убедиться через кросс-валидацию, что удаление id+"мусорных" признаков и добавление Balanced действительно помогло.

In [20]:
# Запускаем кросс-валидацию на 3 фолда (для скорости)
scores_check = cross_val_score(
    model_check,
    x_train_clean,
    y_train,
    cv=3,
    scoring='roc_auc'
)

0:	learn: 0.6851454	total: 3.02s	remaining: 50m 14s
100:	learn: 0.5983418	total: 4m 1s	remaining: 35m 51s
200:	learn: 0.5930030	total: 8m 5s	remaining: 32m 10s
300:	learn: 0.5892344	total: 12m 16s	remaining: 28m 29s
400:	learn: 0.5861137	total: 16m 18s	remaining: 24m 21s
500:	learn: 0.5837281	total: 20m 28s	remaining: 20m 23s
600:	learn: 0.5817648	total: 24m 28s	remaining: 16m 15s
700:	learn: 0.5799196	total: 28m 37s	remaining: 12m 12s
800:	learn: 0.5783183	total: 32m 51s	remaining: 8m 9s
900:	learn: 0.5768937	total: 36m 56s	remaining: 4m 3s
999:	learn: 0.5755026	total: 41m 1s	remaining: 0us
0:	learn: 0.6845942	total: 3.15s	remaining: 52m 29s
100:	learn: 0.5939042	total: 4m 3s	remaining: 36m 6s
200:	learn: 0.5878152	total: 8m 13s	remaining: 32m 40s
300:	learn: 0.5846031	total: 12m 16s	remaining: 28m 31s
400:	learn: 0.5817868	total: 16m 18s	remaining: 24m 21s
500:	learn: 0.5793904	total: 20m 20s	remaining: 20m 15s
600:	learn: 0.5773477	total: 24m 23s	remaining: 16m 11s
700:	learn: 0.575

In [21]:
# 1. результаты всех 3 тестов
print("Результаты на каждом фолде:", scores_check)

# 2. среднее значение
print(f"Средний ROC-AUC: {scores_check.mean():.4f}")

# 3. разброс (стабильность модели)
print(f"Отклонение: +/- {scores_check.std():.4f}")

Результаты на каждом фолде: [0.77344172 0.76251519 0.73555463]
Средний ROC-AUC: 0.7572
Отклонение: +/- 0.0159


In [43]:
# Если ROC-AUC устраивает, то обучаем ОДИН РАЗ на всех очищенных данных
model_final = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    auto_class_weights='Balanced',
    cat_features=cat_features_clean,
    verbose=100
)

model_final.fit(x_train_clean, y_train) # Вот он, финальный fit

# Сохраняем результат, который будем использовать в жизни
model_final.save_model('loan_model_final_v2.cbm')

0:	learn: 0.6846064	total: 4.03s	remaining: 1h 7m 7s
100:	learn: 0.5931697	total: 6m 15s	remaining: 55m 45s
200:	learn: 0.5863500	total: 12m 36s	remaining: 50m 9s
300:	learn: 0.5832683	total: 19m	remaining: 44m 7s
400:	learn: 0.5807340	total: 25m 1s	remaining: 37m 22s
500:	learn: 0.5787314	total: 31m 15s	remaining: 31m 7s
600:	learn: 0.5770309	total: 37m 36s	remaining: 24m 58s
700:	learn: 0.5756187	total: 44m 5s	remaining: 18m 48s
800:	learn: 0.5743635	total: 50m 36s	remaining: 12m 34s
900:	learn: 0.5732440	total: 56m 50s	remaining: 6m 14s
999:	learn: 0.5721902	total: 1h 2m 59s	remaining: 0us


достаю тестовую выборку

In [44]:
test = pd.read_csv('shift_ml_2026_test.csv')

In [45]:
test.shape

(134531, 108)

In [46]:
test.duplicated().sum()

np.int64(0)

In [47]:
# Сохраняем ID отдельно, чтобы потом соединить с результатом
client_ids = test['id']

In [48]:
# Подготавливаю данные в тесте:
# 1. Список колонок, которые мы РЕАЛЬНО использовали при обучении
features_used = model_final.feature_names_

# 2. Оставляем в тесте только эти колонки. Это автоматически удалит id, нулевые признаки и любой другой "мусор"
test_clean = test[features_used]


In [49]:
test_clean.shape   # должно остаться колонок 83 - 13 (id, не важные)

(134531, 70)

In [50]:
test_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 134531 entries, 0 to 134530
Data columns (total 70 columns):
 #   Column                                             Non-Null Count   Dtype  
---  ------                                             --------------   -----  
 0   сумма_займа                                        134531 non-null  float64
 1   срок_займа                                         134531 non-null  object 
 2   процентная_ставка                                  134531 non-null  float64
 3   аннуитет                                           134531 non-null  float64
 4   рейтинг                                            134531 non-null  object 
 5   допрейтинг                                         134531 non-null  object 
 6   профессия_заемщика                                 125795 non-null  object 
 7   стаж                                               126528 non-null  object 
 8   владение_жильем                                    134531 non-null  object

In [ ]:
test_clean = test_clean.fillna(x_train.median(numeric_only=True))   # пропуски в числовых признаках заполняем медианой

In [52]:
test_clean['профессия_заемщика'] = test_clean['профессия_заемщика'].fillna('unknown')
test_clean['стаж'] = test_clean['стаж'].fillna('unknown')        # остались пропуски в двух кат признаках, заполняем 'unknown'

In [53]:
test_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 134531 entries, 0 to 134530
Data columns (total 70 columns):
 #   Column                                             Non-Null Count   Dtype  
---  ------                                             --------------   -----  
 0   сумма_займа                                        134531 non-null  float64
 1   срок_займа                                         134531 non-null  object 
 2   процентная_ставка                                  134531 non-null  float64
 3   аннуитет                                           134531 non-null  float64
 4   рейтинг                                            134531 non-null  object 
 5   допрейтинг                                         134531 non-null  object 
 6   профессия_заемщика                                 134531 non-null  object 
 7   стаж                                               134531 non-null  object 
 8   владение_жильем                                    134531 non-null  object

In [55]:
# 3. Теперь предсказание сработает идеально
result_proba = model_final.predict_proba(test_clean)[:, 1]

In [56]:
result_proba

array([0.24317169, 0.13630745, 0.31246654, ..., 0.54545574, 0.57425802,
       0.45340933])

In [57]:
# Убедиться, что количество ID и количество вероятностей совпадают
if len(client_ids) == len(result_proba):
    # Создаем DataFrame из двух колонок
    results_df = pd.DataFrame({
        'client_id': client_ids,
        'probability': result_proba
    })

    # Сортируем по вероятности, чтобы увидеть самых рискованных клиентов
    #results_df = results_df.sort_values(by='probability', ascending=False)

    # Выводим первые 10 строк
    #print(results_df.head(10))

    # Сохраняем результат в файл
    results_df.to_csv('predictions_with_ids.csv', index=False)
else:
    print("Ошибка: Количество ID клиентов не совпадает с количеством предсказаний.")

In [61]:
# Переименовываем колонки в соответствии с требованиями codabench
submission = results_df.rename(columns={
    'client_id': 'ИДЕНТИФИКАТОР',
    'probability': 'Проба'
})

# Сохраняем в CSV без индекса
submission.to_csv('submission.csv', index=False)

print("Файл submission.csv готов.")

Файл submission.csv готов.


In [62]:
submission

,ИДЕНТИФИКАТОР,Проба
0,85540387,0.243172
1,28112500,0.136307
2,65731570,0.312467
3,65874747,0.818722
4,57893355,0.663113
...,...,...
134526,72082596,0.844500
134527,484702,0.104857
134528,73273241,0.545456
134529,50466330,0.574258


In [63]:
import zipfile
import os

csv_file_name = 'submission.csv'
zip_file_name = 'submission.zip'

# Создаем ZIP-архив
with zipfile.ZipFile(zip_file_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(csv_file_name, os.path.basename(csv_file_name))

print(f"Файл '{csv_file_name}' успешно заархивирован в '{zip_file_name}'")

Файл 'submission.csv' успешно заархивирован в 'submission.zip'
